# O‘zMSt 285:2024, 1-algoritm: kriptotahlil xulosalarini misollar bilan kuzatish

Ushbu daftar «SBZ o‘zgartirishining kriptotahlili» hisobotidagi 13 ta xulosaning har birini aniq misolda,
akslantirishlar ketma-ketligi bo‘yicha qadamma-qadam kuzatadi:
chiziqli tahlil — 5 ta, differensial tahlil — 4 ta, algebraik tahlil — 4 ta xulosa.

**Dastlabki ma’lumotlar:** standartning A ilovasidagi xeshlash kaliti va 64 baytli xabar.
Daftar `algoritm.py`, `komponentlar.py`, `differensial.py` va `algebraik.py` modullari bilan bir papkada ishga tushiriladi.
`R` lug‘atiga barcha natijalar yig‘iladi va oxirida `misollar_natijalari.json` fayliga yoziladi.

In [1]:
import numpy as np, itertools, math, json
from algoritm import *
from komponentlar import *
np.set_printoptions(linewidth=140)

KALIT = np.array(list(KAT_KALIT))          # 32 bayt
XABAR = np.array(list(KAT_XABAR))          # 64 bayt
BLOK1, BLOK2 = XABAR[:32], XABAR[32:]
R = {}

def hx(a):
    """4x8 massivni o‘n oltilik jadval ko‘rinishida (satrlar ro‘yxati)."""
    a = np.asarray(a).reshape(4, 8); w = 2 if a.max() > 15 else 1
    return [' '.join(f'{int(v):0{w}x}' for v in row) for row in a]
def bx(a):
    """4x8 massivning 0-qatlami (har elementning eng kichik biti)."""
    a = np.asarray(a).reshape(4, 8) % 2
    return [' '.join(str(int(v)) for v in row) for row in a]
def chiqar(nom, a, f=hx):
    print(nom); [print('   ', s) for s in f(a)]
def iz(c, holat, blok, kalit):
    """Bitta iteratsiya izi: [(amal, raund, holat, ke), ...]."""
    tr = []; K = kalit[None]
    iteration(holat[None].copy(), blok[None].copy(), K.copy(), K, c, trace=tr)
    return [(n, j, h[0], k[0]) for n, j, h, k in tr]
def kirish_bloklari(xabar, p):
    """Asosiy qism bloklari + uzunlik bloki + NY bloki (birliklar: bayt yoki yarim bayt)."""
    bl = 32 if p == 256 else 16; nb = len(xabar) // bl
    bloks = [xabar[i*bl:(i+1)*bl] for i in range(nb)]
    ny = sum(int.from_bytes(bytes(b.tolist()), 'big') for b in bloks) % 2**(8*bl)
    return [to_units(b, p) for b in bloks] + [to_units(np.array(list((len(xabar)*8).to_bytes(bl, 'big'))), p),
                                              to_units(np.array(list(ny.to_bytes(bl, 'big'))), p)]
K16 = to_units(np.array(list(KAT_KALIT[:16])), 16)   # 128 bitli rejim kaliti (32 ta yarim bayt)

print("Nazorat qiymatlari:"); print("hammasi mos" if nazorat() else "XATO")

Nazorat qiymatlari:
  p=256, std: MOS
  p=256, sbz: MOS
  p= 16, std: MOS
  p= 16, sbz: MOS
hammasi mos


## 1. Chiziqli kriptotahlil

### 1-xulosa. Yutishsiz raundlar 0- va 1-bit qatlamlarni c ≈ 1 bilan o‘tkazadi

Misol: SBZ-256, A ilova kaliti va xabari, 1-iteratsiya. 0-nuqtadagi yutishdan keyingi holatning 0-qatlami
`h0` dan boshlab har bir akslantirishdan keyin 0-qatlamni nazariy bashorat qilamiz:
Aralash → `M(k0)·h0`, Qo‘sh → o‘zgarmaydi, SurHolat → `P·h0`, TuzilmaKalit → holatga ta’sir qilmaydi.

In [2]:
c = Cfg(256, 'sbz'); tr = iz(c, BLOK1, BLOK2, KALIT)
Mk, Pm, L = M_aralash(KALIT), P_SUR, L_raund(KALIT)
h_bz0 = [t for t in tr if t[0] == 'BZ0'][0][2]
chiqar("0-nuqtadagi yutishdan (SBZ) keyingi holat:", h_bz0)
chiqar("uning 0-qatlami h0:", h_bz0, bx)
pred = h_bz0 % 2; qadamlar = []
for nom, j, h, ke in tr:
    if j != 1: continue
    if nom == 'Aralash': pred = (Mk @ pred) % 2
    if nom == 'Sur': pred = (Pm @ pred) % 2
    mos = int((pred == h % 2).sum())
    qadamlar.append(dict(amal=nom, holat=hx(h), qatlam0=bx(h), bashorat=bx(pred), mos=mos))
    print(f"\n1-raund, {nom}: 0-qatlamda bashorat bilan mos bitlar {mos}/32")
    chiqar("  holat:", h); chiqar("  0-qatlam (haqiqiy):", h, bx)
ke_t = [t for t in tr if t[0] == 'Tuz' and t[1] == 1][0][3]
ke_ok = bool(((ke_t % 2) == (KALIT % 2)).all())
print("\nTuzilmaKalit dan keyin ke 0-qatlami k 0-qatlamiga tengmi:", ke_ok)
pred = h_bz0 % 2; raund = []
for j in range(1, 11):
    pred = (L @ pred) % 2
    act = [t for t in tr if t[1] == j][-1][2] % 2
    raund.append(int((pred == act).sum()))
print("\nRaundlar oxirida L^j·h0 bashorati bilan mos bitlar (j = 1..10):", raund)
R['L1'] = dict(h_bz0=hx(h_bz0), h0=bx(h_bz0), qadamlar=qadamlar, raund_mos=raund, rangL=rank2(L), ke_ok=ke_ok)

0-nuqtadagi yutishdan (SBZ) keyingi holat:
    bf 76 4e 77 0c 06 30 ef
    33 e9 26 95 6d fe b0 c6
    b3 25 41 ee 45 32 c4 e0
    7c 5f 52 69 35 a4 fb 0f
uning 0-qatlami h0:
    1 0 0 1 0 0 0 1
    1 1 0 1 1 0 0 0
    1 1 1 0 1 0 0 0
    0 1 0 1 1 0 1 1

1-raund, Aralash: 0-qatlamda bashorat bilan mos bitlar 32/32
  holat:
    c0 ec 3b 45 5b 59 e4 87
    99 62 e3 f8 56 5f 75 e5
    da 0c 81 f4 44 3d 55 5b
    1d 07 ff 4f 85 28 c3 07
  0-qatlam (haqiqiy):
    0 0 1 1 1 1 0 1
    1 0 1 0 0 1 1 1
    0 0 1 0 0 1 1 1
    1 1 1 1 1 0 1 1

1-raund, Qosh1: 0-qatlamda bashorat bilan mos bitlar 32/32
  holat:
    40 84 9f 49 75 9b 54 71
    ef fe d3 58 82 d1 7d e9
    1a 4c 9b f4 1c 2b 4f 37
    4d 17 6d 55 9b 28 9d 3f
  0-qatlam (haqiqiy):
    0 0 1 1 1 1 0 1
    1 0 1 0 0 1 1 1
    0 0 1 0 0 1 1 1
    1 1 1 1 1 0 1 1

1-raund, Sur: 0-qatlamda bashorat bilan mos bitlar 32/32
  holat:
    9b 4f ef 9f 4d 9b 82 54
    71 28 37 fe 49 17 f4 d1
    7d 40 9d 1a d3 75 6d 1c
    2b e9 84 3f 4c 58 9b 5

Endi 1-bit qatlamni tekshiramiz. Haqiqiy algoritmdagi 0-nuqta Qo‘sh o‘zgartirishining holat[0,0] elementi
uchun A, B parametrlarini hisoblab, bitta element akslantirishini barcha 256 kirish bo‘yicha tekshiramiz.

In [3]:
from algoritm import prododd, dK, step
def qosh_batafsil(h, hn, Rv, p=256):
    """Qo‘sh(holat, holatn) ning holat qismi uchun barcha oraliq kattaliklar (algoritm.qosh bilan bir xil)."""
    c = Cfg(p); H, N = h[None], hn[None]; Rr = np.array([[Rv]])
    J1h, J1hn = int(h.sum() % p), int(hn.sum() % p)
    Ph, Phn = int(prododd(H, p)[0]), int(prododd(N, p)[0])
    K1h = (Ph + int(dK(np.array([Ph]), np.array([J1hn]), p)[0])) % p
    K1hn = (Phn + int(dK(np.array([Phn]), np.array([J1h]), p)[0])) % p
    A1 = (J1hn + 2 * hn + (0 if J1hn % 2 else 1)) % p; B1 = (K1h - 2 * h) % p
    x1 = step(H, A1[None], B1[None], Rr, c)[0]
    A1n = (J1h + 2 * h + (0 if J1h % 2 else 1)) % p; B1n = (K1hn - 2 * hn) % p
    xn1 = step(N, A1n[None], B1n[None], Rr, c)[0]
    J2h, J2hn = int(x1.sum() % p), int(xn1.sum() % p)
    P2h = int(prododd(x1[None], p)[0])
    K2h = (P2h + int(dK(np.array([P2h]), np.array([J2hn]), p)[0])) % p
    A2 = (J1hn + J2hn + 2 * xn1 + (0 if (J1hn + J2hn) % 2 else 1)) % p; A2 = np.where(A2 == 0, 1, A2)
    B2 = (K1h * K2h - 2 * x1) % p
    x2 = step(x1[None], A2[None], B2[None], Rr, c)[0]; x2 = np.where(x2 == 0, (A2 + B2) % p, x2)
    x3 = step(x2[None], A2[None], B2[None], Rr, c)[0]
    return dict(J1h=J1h, J1hn=J1hn, K1h=K1h, A1=A1, B1=B1, x1=x1, A2=A2, B2=B2, x2=x2, x3=x3, J2hn=J2hn, K2h=K2h)

Rv = int(KALIT[0]); q = qosh_batafsil(BLOK1, BLOK2, Rv)
print("Tekshiruv: qosh_batafsil natijasi algoritm.qosh bilan bir xilmi:",
      bool((q['x3'] == qosh(BLOK1[None], BLOK2[None], np.array([Rv]), Cfg(256))[0][0]).all()))
e = 0   # holat[0,0]
A1, B1, A2, B2 = int(q['A1'][e]), int(q['B1'][e]), int(q['A2'][e]), int(q['B2'][e])
print(f"R = {Rv:#04x}; holat[0,0] uchun A1 = {A1}, B1 = {B1}, A2 = {A2}, B2 = {B2}")
f = qosh_elem(A1, B1, A2, B2, Rv); x = np.arange(256)
y0_ok = int(((f & 1) == (x & 1)).sum())
best = max(((d, int((((f >> 1) & 1) == (((x >> 1) & 1) ^ (d * (x & 1)))).sum())) for d in (0, 1)), key=lambda t: t[1])
print(f"y0 = x0 tengligi: {y0_ok}/256 kirishda bajariladi")
print(f"y1 = x1 ⊕ {best[0]}·x0 tengligi: {best[1]}/256 kirishda bajariladi")
W = lat(f, 8); bitc = [round(float(np.abs(W[1 << j]).max() / 256), 3) for j in range(8)]
print("Har bir chiqish biti uchun eng yaxshi |c|:", bitc)
R['L1'].update(param=dict(R=Rv, A1=A1, B1=B1, A2=A2, B2=B2), y0_ok=y0_ok, y1=list(best), bitc=bitc)

Tekshiruv: qosh_batafsil natijasi algoritm.qosh bilan bir xilmi: True
R = 0xc8; holat[0,0] uchun A1 = 73, B1 = 201, A2 = 47, B2 = 245
y0 = x0 tengligi: 256/256 kirishda bajariladi
y1 = x1 ⊕ 1·x0 tengligi: 256/256 kirishda bajariladi
Har bir chiqish biti uchun eng yaxshi |c|: [1.0, 1.0, 0.508, 0.75, 0.883, 0.562, 0.523, 0.258]


### 2-xulosa. Standart-128 da 0-qatlam butun xesh davomida c = 1 bilan kuzatiladi

Misol: Standart-128 (p = 16), kalit sifatida A ilova kalitining birinchi 16 bayti, 64 baytli A ilova xabari
(4 ta blok + uzunlik + NY = 6 ta kirish bloki, 5 ta iteratsiya). Bashorat qoidasi: har raundda `h0 ← L·h0`,
yutish nuqtalarida `h0 ← h0 ⊕ blok0`.

In [4]:
c16 = Cfg(16, 'std'); kir = kirish_bloklari(XABAR, 16); M16 = M_aralash(K16)
holat = kir[0].copy(); ke = K16.copy(); pred = holat % 2; jurnal = []
for i in range(1, len(kir)):
    tr = []; K = K16[None]
    h_out, ke_out = iteration(holat[None], kir[i][None], ke[None].copy(), K, c16, trace=tr)
    b0 = kir[i] % 2; mos_iter = []
    for nom, j, h, kk in tr:
        if nom in ('BZ0', 'BZ8', 'BZ9'): pred = pred ^ b0
        if nom == 'Aralash': pred = (M16 @ pred) % 2
        if nom == 'Sur': pred = (P_SUR @ pred) % 2
        mos_iter.append(int((pred == h[0] % 2).sum()))
    jurnal.append(dict(iteratsiya=i, amallar=len(mos_iter), min_mos=min(mos_iter)))
    print(f"{i}-iteratsiya: {len(mos_iter)} ta akslantirishdan keyin bashorat bilan mos bitlar (minimum) = {min(mos_iter)}/32")
    holat, ke = h_out[0], ke_out[0]
xq = bytes(int(holat[2*i] << 4 | holat[2*i+1]) for i in range(16)).hex()
print("\nXesh qiymat:", xq, "(nazorat:", KAT[(16, 'std')] == xq, ")")
chiqar("Xesh qiymatning 0-qatlami (haqiqiy):", holat, bx)
chiqar("Bashorat qilingan 0-qatlam:", pred, bx)
R['L2'] = dict(jurnal=jurnal, xesh=xq, haqiqiy=bx(holat), bashorat=bx(pred), mos=int((pred == holat % 2).sum()))

1-iteratsiya: 55 ta akslantirishdan keyin bashorat bilan mos bitlar (minimum) = 32/32
2-iteratsiya: 55 ta akslantirishdan keyin bashorat bilan mos bitlar (minimum) = 32/32
3-iteratsiya: 55 ta akslantirishdan keyin bashorat bilan mos bitlar (minimum) = 32/32
4-iteratsiya: 55 ta akslantirishdan keyin bashorat bilan mos bitlar (minimum) = 32/32
5-iteratsiya: 55 ta akslantirishdan keyin bashorat bilan mos bitlar (minimum) = 32/32

Xesh qiymat: e9eaccdd4ccce2d229228c864989eb9f (nazorat: True )
Xesh qiymatning 0-qatlami (haqiqiy):
    0 1 0 0 0 0 1 1
    0 0 0 0 0 0 1 0
    0 1 0 0 0 0 0 0
    0 1 0 1 0 1 1 1
Bashorat qilingan 0-qatlam:
    0 1 0 0 0 0 1 1
    0 0 0 0 0 0 1 0
    0 1 0 0 0 0 0 0
    0 1 0 1 0 1 1 1


### 3-xulosa. Standart-256: faol bayt boshiga c ≤ 2⁻²

Avval B ilovadagi Z jadvali uchun eng yaxshi approksimatsiyani barcha 256 kirishda tekshiramiz, so‘ng
uni haqiqiy yutish nuqtasida (Qo‘sh dan keyingi holatn bayti → yangi holatning quyi yarim bayti) o‘lchaymiz.
Keyin A ilova kaliti uchun yutish qatlamlari bo‘yicha minimal faol baytlarni topamiz.

In [5]:
x = np.arange(256); z = Z[x]
lhs = (z & 1) ^ ((z >> 2) & 1); rhs = (x & 1) ^ ((x >> 2) & 1) ^ ((x >> 6) & 1) ^ ((x >> 7) & 1)
agree = int((lhs == rhs).sum()); cZ = 2 * agree / 256 - 1
print(f"z0 ⊕ z2 = x0 ⊕ x2 ⊕ x6 ⊕ x7 :  {agree}/256 kirishda bajariladi,  c = {cZ}")
rng = np.random.default_rng(1); N = 20000; cs = Cfg(256, 'std')
H = rng.integers(0, 256, (N, 32)); B = rng.integers(0, 256, (N, 32)); Kk = kalitlar(rng, N, 256)
h1, hn1 = qosh(H, B, Kk[:, 0], cs); out = bzf(h1, hn1, cs)
lo = out & 15; L_ = (lo & 1) ^ ((lo >> 2) & 1); Rr = (hn1 & 1) ^ ((hn1 >> 2) & 1) ^ ((hn1 >> 6) & 1) ^ ((hn1 >> 7) & 1)
c_meas = float(2 * (L_[:, 0] == Rr[:, 0]).mean() - 1)
print(f"Algoritmdagi yutish nuqtasida (bayt 0, {N} ta tasodifiy holat): o‘lchangan c = {c_meas:.3f}")

def zanjir(Lm):
    """Qatlamlar 0, 8, 9, 0', 8', 9' bo‘yicha minimal faol baytlar va eng yaxshi niqob pozitsiyalari."""
    L8 = np.linalg.matrix_power(Lm, 8) % 2; maps = [L8, Lm, Lm, L8, Lm]
    best = {kk: (99, None) for kk in range(1, 7)}
    cands = [np.eye(32, dtype=np.int64)[list(cc)].sum(0) % 2 for w in (1, 2) for cc in itertools.combinations(range(32), w)]
    for end in range(1, 6):
        for u in cands:
            tot = int(u.sum()); cur = u; n = 1; path = [tuple(int(v) for v in np.nonzero(u)[0])]
            if tot < best[1][0]: best[1] = (tot, list(path))
            for m in range(end - 1, -1, -1):
                cur = (maps[m].T @ cur) % 2
                if cur.sum() == 0: break
                tot += int(cur.sum()); n += 1; path.append(tuple(int(v) for v in np.nonzero(cur)[0]))
                if tot < best[n][0]: best[n] = (tot, list(path))
    return best
zA = zanjir(L)
print(f"\nA ilova kaliti uchun L ning [0,3] satri: {[pos3 for pos3 in np.nonzero(L[3])[0]]} -> 0-qatlamdagi holat[0,3] biti har raundda o‘zgarmay o‘tadi")
print("A ilova kaliti: qatlamlar soni -> minimal faol baytlar; Standart-256 chegarasi log2|c| <= -2·a")
for kk in range(1, 7):
    a, path = zA[kk]
    if path: print(f"  {kk} qatlam: a = {a:2d} -> log2|c| <= {-2*a:4d};  faol pozitsiyalar (oxirgi qatlamdan orqaga): {[list(p) for p in path]}")
R['L3'] = dict(agree=agree, cZ=cZ, c_meas=round(c_meas, 3),
               zanjir={kk: (zA[kk][0], [list(p) for p in zA[kk][1]] if zA[kk][1] else None) for kk in zA})

z0 ⊕ z2 = x0 ⊕ x2 ⊕ x6 ⊕ x7 :  160/256 kirishda bajariladi,  c = 0.25
Algoritmdagi yutish nuqtasida (bayt 0, 20000 ta tasodifiy holat): o‘lchangan c = 0.254

A ilova kaliti uchun L ning [0,3] satri: [np.int64(3)] -> 0-qatlamdagi holat[0,3] biti har raundda o‘zgarmay o‘tadi
A ilova kaliti: qatlamlar soni -> minimal faol baytlar; Standart-256 chegarasi log2|c| <= -2·a
  1 qatlam: a =  1 -> log2|c| <=   -2;  faol pozitsiyalar (oxirgi qatlamdan orqaga): [[0]]
  2 qatlam: a =  2 -> log2|c| <=   -4;  faol pozitsiyalar (oxirgi qatlamdan orqaga): [[3], [3]]
  3 qatlam: a =  3 -> log2|c| <=   -6;  faol pozitsiyalar (oxirgi qatlamdan orqaga): [[3], [3], [3]]
  4 qatlam: a =  4 -> log2|c| <=   -8;  faol pozitsiyalar (oxirgi qatlamdan orqaga): [[3], [3], [3], [3]]
  5 qatlam: a =  5 -> log2|c| <=  -10;  faol pozitsiyalar (oxirgi qatlamdan orqaga): [[3], [3], [3], [3], [3]]
  6 qatlam: a =  6 -> log2|c| <=  -12;  faol pozitsiyalar (oxirgi qatlamdan orqaga): [[3], [3], [3], [3], [3], [3]]


### 4-xulosa. SBZ: faol bayt boshiga c ≤ 2⁻⁶

SBZ approksimatsiyasi `y0 ⊕ ⟨2D,h⟩ ⊕ ⟨1A,n⟩ = 1` ni barcha 65536 ta (h, n) juftlikda tekshiramiz,
so‘ng A ilova kaliti va «eng yomon» kalit uchun yutish qatlamlari bo‘yicha chegaralarni hisoblaymiz.

In [6]:
hh = np.arange(65536) >> 8; nn = np.arange(65536) & 255; y = S[hh ^ S[nn]]
par = lambda v, m: PAR8[v & m]
rel = (y & 1) ^ par(hh, 0x2D) ^ par(nn, 0x1A)
ones = int(rel.sum()); cS = 1 - 2 * ones / 65536
WS = lat(S, 8)
print(f"y0 ⊕ <2D,h> ⊕ <1A,n> ifodasi {ones}/65536 juftlikda 1 ga teng;  korrelyatsiya = {cS} = −2^{math.log2(-cS):.0f}")
print(f"c_S(2D→01) = {WS[1,0x2D]/256},  c_S(1A→2D) = {WS[0x2D,0x1A]/256},  ko‘paytma = {WS[1,0x2D]*WS[0x2D,0x1A]/65536}")
print("\nA ilova kaliti: SBZ chegarasi log2|c| <= −6·a")
for kk in range(1, 7):
    a = zA[kk][0]
    if a < 99: print(f"  {kk} qatlam: a = {a:2d} -> log2|c| <= {-6*a}")
rng = np.random.default_rng(9); yomon = None
for t in range(300):
    kk_ = kalitlar(rng, 1, 256)[0]; zz = zanjir(L_raund(kk_))
    if zz[4][0] >= 14: yomon = (kk_, zz); break
if yomon:
    print("\nOdatiy kalit (4 ta qatlamda >= 14 faol bayt):", bytes(int(v) for v in yomon[0]).hex())
    for kk in range(1, 7):
        a = yomon[1][kk][0]
        if a < 99: print(f"  {kk} qatlam: a = {a:2d} -> log2|c| <= {-6*a};  pozitsiyalar: {[list(p) for p in yomon[1][kk][1]]}")
R['L4'] = dict(ones=ones, c=cS, zanjir_A={kk: zA[kk][0] for kk in zA},
               yomon_kalit=bytes(int(v) for v in yomon[0]).hex() if yomon else None,
               yomon={kk: (yomon[1][kk][0], [list(p) for p in yomon[1][kk][1]] if yomon[1][kk][1] else None) for kk in yomon[1]} if yomon else None)

y0 ⊕ <2D,h> ⊕ <1A,n> ifodasi 33280/65536 juftlikda 1 ga teng;  korrelyatsiya = -0.015625 = −2^-6
c_S(2D→01) = -0.125,  c_S(1A→2D) = 0.125,  ko‘paytma = -0.015625

A ilova kaliti: SBZ chegarasi log2|c| <= −6·a
  1 qatlam: a =  1 -> log2|c| <= -6
  2 qatlam: a =  2 -> log2|c| <= -12
  3 qatlam: a =  3 -> log2|c| <= -18
  4 qatlam: a =  4 -> log2|c| <= -24
  5 qatlam: a =  5 -> log2|c| <= -30
  6 qatlam: a =  6 -> log2|c| <= -36

Odatiy kalit (4 ta qatlamda >= 14 faol bayt): 6adef6491d9aaac7a4b7eaeaeddcb8eb0306cc6fbf7ce610ac0125d44afb59c8
  1 qatlam: a =  1 -> log2|c| <= -6;  pozitsiyalar: [[0]]
  2 qatlam: a =  2 -> log2|c| <= -12;  pozitsiyalar: [[11], [27]]
  3 qatlam: a =  4 -> log2|c| <= -24;  pozitsiyalar: [[11], [27], [4, 13]]
  4 qatlam: a = 18 -> log2|c| <= -108;  pozitsiyalar: [[5], [18, 27], [4, 6, 13, 14, 22], [2, 3, 8, 10, 13, 16, 20, 23, 29, 30]]
  5 qatlam: a = 34 -> log2|c| <= -204;  pozitsiyalar: [[16, 27], [4, 13, 14, 15], [8, 11, 12, 15, 16, 18, 21, 22, 23, 24, 28], [3,

### 5-xulosa. Chiqish 0-qatlamidagi c = 1 munosabatlar SBZ ga bog‘liq emas

Oxirgi (10-) raund yutishdan keyin bajariladi, shuning uchun xesh qiymatning 0-qatlami `L·x0` ga teng.
`wᵀ·L = 0` bo‘lgan har bir `w` vektor chiqishda `w·h0 = 0` munosabatini beradi. A ilova kaliti uchun bu
munosabatlarni topib, 1000 ta tasodifiy xabarda ikkala variant uchun tekshiramiz.

In [7]:
def chap_yadro(A):
    """wᵀA = 0 tenglamasining GF(2) dagi bazisi."""
    rows = A.T.copy() % 2; n = rows.shape[1]; piv = []; r = 0
    for col in range(n):
        p_ = [i for i in range(r, rows.shape[0]) if rows[i, col]]
        if not p_: continue
        rows[[r, p_[0]]] = rows[[p_[0], r]]
        for i in range(rows.shape[0]):
            if i != r and rows[i, col]: rows[i] ^= rows[r]
        piv.append(col); r += 1
    basis = []
    for fcol in [cl for cl in range(n) if cl not in piv]:
        v = np.zeros(n, dtype=np.int64); v[fcol] = 1
        for i, pc in enumerate(piv): v[pc] = rows[i, fcol]
        basis.append(v)
    return basis
baz = chap_yadro(L)
nol_satr = [int(i) for i in np.nonzero(L.sum(1) == 0)[0]]
pos = lambda i: f"[{i//8},{i%8}]"
print(f"rang(L) = {rank2(L)}, munosabatlar soni = {len(baz)}, nol satrlar (doimiy nol bitlar) = {[pos(i) for i in nol_satr]}")
for v in baz: print("   " + " ⊕ ".join(pos(i) for i in np.nonzero(v)[0]) + " = 0")
rng = np.random.default_rng(3); msgs = rng.integers(0, 256, (1000, 64)); Kt = np.tile(KALIT, (1000, 1))
tek = {}
for bz in ('std', 'sbz'):
    out = hash_msgs(msgs, Kt, Cfg(256, bz)) % 2
    tek[bz] = [float(((out @ v) % 2 == 0).mean()) for v in baz]
    print(f"{bz}: har bir munosabatning 1000 ta xabarda bajarilish ulushi = {tek[bz]}")
R['L5'] = dict(rang=rank2(L), munosabatlar=[[pos(i) for i in np.nonzero(v)[0]] for v in baz], nol_satr=[pos(i) for i in nol_satr], tekshiruv=tek)

rang(L) = 24, munosabatlar soni = 8, nol satrlar (doimiy nol bitlar) = []
   [0,4] ⊕ [1,5] = 0
   [0,1] ⊕ [1,7] = 0
   [1,3] ⊕ [2,1] = 0
   [0,6] ⊕ [1,0] ⊕ [2,5] ⊕ [2,7] = 0
   [0,3] ⊕ [1,4] ⊕ [3,2] = 0
   [0,2] ⊕ [0,4] ⊕ [0,5] ⊕ [1,3] ⊕ [1,4] ⊕ [1,6] ⊕ [2,3] ⊕ [2,4] ⊕ [2,6] ⊕ [3,4] = 0
   [0,7] ⊕ [1,1] ⊕ [2,2] ⊕ [3,6] = 0
   [0,5] ⊕ [1,3] ⊕ [3,7] = 0
std: har bir munosabatning 1000 ta xabarda bajarilish ulushi = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
sbz: har bir munosabatning 1000 ta xabarda bajarilish ulushi = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


## 2. Differensial kriptotahlil

### 1-xulosa. Yutishsiz raundlar MSB izlarini ≈0,85–0,94 ehtimollik bilan o‘tkazadi

Misol: SBZ-256, A ilova kaliti. 0-nuqtadagi yutishdan keyingi holat (`h_bz0`) va undan holat[0,0] baytining
MSB biti bilan farq qiladigan ikkinchi holat olinadi. Har bir akslantirishdan keyin ayirma kuzatiladi.

In [8]:
from differensial import raundlar_yutishsiz
def juft_iz(c, h, h2, blok, kalit):
    K = kalit[None]; t1, t2 = [], []
    raundlar_yutishsiz(h[None].copy(), blok[None], K.copy(), K, c, t1)
    raundlar_yutishsiz(h2[None].copy(), blok[None], K.copy(), K, c, t2)
    return [(a[0], a[1], a[2][0] ^ b[2][0], a[3][0] ^ b[3][0]) for a, b in zip(t1, t2)]
msb_only = lambda d, dk, m=0x80: bool(((d == 0) | (d == m)).all() and ((dk == 0) | (dk == m)).all())
c = Cfg(256, 'sbz'); h2 = h_bz0.copy(); h2[0] ^= 0x80
jr = juft_iz(c, h_bz0, h2, BLOK2, KALIT); D1qadam = []
for nom, j, d, dk in jr:
    if j > 2: break
    D1qadam.append(dict(amal=nom, raund=j, dh=hx(d), faol=int((d != 0).sum()), faol_ke=int((dk != 0).sum()), msb=msb_only(d, dk)))
    print(f"\n{j}-raund, {nom}: holatda {int((d!=0).sum())} ta, ke da {int((dk!=0).sum())} ta faol bayt; faqat MSB: {msb_only(d, dk)}")
    chiqar("  Δholat:", d)
xul = []
for j in range(1, 11):
    d, dk = [(t[2], t[3]) for t in jr if t[1] == j][-1]
    xul.append(dict(raund=j, faol=int((d != 0).sum()), faol_ke=int((dk != 0).sum()), msb=msb_only(d, dk)))
print("\nRaundlar oxirida (raund, faol holat, faol ke, faqat MSB):", [(r_['raund'], r_['faol'], r_['faol_ke'], r_['msb']) for r_ in xul])
rng = np.random.default_rng(9); N = 3000; Kk = kalitlar(rng, N, 256)
Hh = rng.integers(0, 256, (N, 32)); Bb = rng.integers(0, 256, (N, 32)); H2 = Hh.copy(); H2[:, 0] ^= 0x80
t1, t2 = [], []; raundlar_yutishsiz(Hh, Bb, Kk.copy(), Kk, c, t1); raundlar_yutishsiz(H2, Bb, Kk.copy(), Kk, c, t2)
stat = []
for (nm, j, a, ka), (_, _, b, kb) in zip(t1, t2):
    if nm == 'Tuz':
        d, dk = a ^ b, ka ^ kb
        stat.append(round(float((((d == 0) | (d == 0x80)).all(1) & ((dk == 0) | (dk == 0x80)).all(1)).mean()), 3))
print("MSB izining saqlanish ehtimolligi, 3000 juftlik (raund 1..10):", stat)
R['D1'] = dict(qadam=D1qadam, xulosa=xul, stat=stat)


1-raund, Aralash: holatda 2 ta, ke da 0 ta faol bayt; faqat MSB: True
  Δholat:
    00 00 00 00 00 00 00 00
    80 00 00 00 00 00 00 00
    80 00 00 00 00 00 00 00
    00 00 00 00 00 00 00 00

1-raund, Qosh1: holatda 2 ta, ke da 0 ta faol bayt; faqat MSB: True
  Δholat:
    00 00 00 00 00 00 00 00
    80 00 00 00 00 00 00 00
    80 00 00 00 00 00 00 00
    00 00 00 00 00 00 00 00

1-raund, Sur: holatda 2 ta, ke da 0 ta faol bayt; faqat MSB: True
  Δholat:
    00 00 80 00 00 00 00 00
    00 00 00 00 00 00 00 00
    00 00 00 80 00 00 00 00
    00 00 00 00 00 00 00 00

1-raund, Qosh2: holatda 2 ta, ke da 0 ta faol bayt; faqat MSB: True
  Δholat:
    00 00 80 00 00 00 00 00
    00 00 00 00 00 00 00 00
    00 00 00 80 00 00 00 00
    00 00 00 00 00 00 00 00

1-raund, Tuz: holatda 2 ta, ke da 0 ta faol bayt; faqat MSB: True
  Δholat:
    00 00 80 00 00 00 00 00
    00 00 00 00 00 00 00 00
    00 00 00 80 00 00 00 00
    00 00 00 00 00 00 00 00

2-raund, Aralash: holatda 4 ta, ke da 0 ta fao

### 2-xulosa. SBZ yutish nuqtasidagi ehtimollikni 2⁻⁶ gacha kamaytiradi va bitta argumentdagi ayirma so‘nmaydi

In [9]:
x = np.arange(256)
col_ok = all(len(set(int(Z[(r << 4) | cc]) for r in range(16))) == 16 for cc in range(16))
row_ok = all(len(set(int(Z[(r << 4) | cc]) for cc in range(16))) == 16 for r in range(16))
print("Z ning har bir ustuni va satri 0..15 ning o‘rin almashtirishi (lotin kvadrati):", col_ok and row_ok)
v96 = [int(v) for v in x if Z[v] == Z[v ^ 0x96]]
print(f"Z(x) = Z(x ⊕ 96) bajariladigan x lar soni: {len(v96)}/256; masalan x = {v96[0]:#04x}: Z(x) = {int(Z[v96[0]]):#x}, Z(x ⊕ 96) = {int(Z[v96[0]^0x96]):#x}")
h0, n0 = int(h_bz0[0]), int(BLOK2[0])
std = [int((Z[h0] << 4) | Z[n0]), int((Z[h0] << 4) | Z[n0 ^ 0x80])]
s1, s2 = int(S[h0 ^ S[n0]]), int(S[h0 ^ S[n0 ^ 0x80]])
print(f"\nMisol: h = {h0:#04x}, n = {n0:#04x}, n' = n ⊕ 80 = {n0^0x80:#04x}")
print(f"  Standart-256:  Z(h)‖Z(n) = {std[0]:#04x},  Z(h)‖Z(n') = {std[1]:#04x}  ->  Δ = {std[0]^std[1]:#04x}")
print(f"  SBZ:           S(h⊕S(n)) = {s1:#04x},  S(h⊕S(n')) = {s2:#04x}  ->  Δ = {s1^s2:#04x}")
cnt0 = 0
for dn in range(1, 256): cnt0 += int((S[hh ^ S[nn]] == S[hh ^ S[nn ^ dn]]).sum())
print(f"\nSBZ: Δh = 0, Δn ≠ 0 bo‘lgan barcha 255·65536 juftlikda so‘nishlar soni = {cnt0}")
both = int((S[hh ^ S[nn]] == S[(hh ^ 0x80) ^ S[nn ^ 0x80]]).sum())
print(f"SBZ: Δh = Δn = 80 da so‘nishlar: {both}/65536 = 2^{math.log2(both/65536):.1f}")
DS = ddt(S, 256); DZ = ddt(Z, 16)
print(f"Maksimal ehtimolliklar: Z — {DZ[1:].max()}/256 = {DZ[1:].max()/256:.3f};  S va SBZ — {DS[1:].max()}/256 = 2^-6;  XOR — 1")
R['D2'] = dict(lotin=col_ok and row_ok, v96=len(v96), v96_misol=[v96[0], int(Z[v96[0]])], h=h0, n=n0, std=std, sbz=[s1, s2],
               sbz_sonish_bir=cnt0, sbz_sonish_80=both, Zmax=int(DZ[1:].max()))

Z ning har bir ustuni va satri 0..15 ning o‘rin almashtirishi (lotin kvadrati): True
Z(x) = Z(x ⊕ 96) bajariladigan x lar soni: 40/256; masalan x = 0x07: Z(x) = 0x8, Z(x ⊕ 96) = 0x8

Misol: h = 0xbf, n = 0x6e, n' = n ⊕ 80 = 0xee
  Standart-256:  Z(h)‖Z(n) = 0x57,  Z(h)‖Z(n') = 0x51  ->  Δ = 0x06
  SBZ:           S(h⊕S(n)) = 0xb7,  S(h⊕S(n')) = 0x88  ->  Δ = 0x3f

SBZ: Δh = 0, Δn ≠ 0 bo‘lgan barcha 255·65536 juftlikda so‘nishlar soni = 0
SBZ: Δh = Δn = 80 da so‘nishlar: 512/65536 = 2^-7.0
Maksimal ehtimolliklar: Z — 40/256 = 0.156;  S va SBZ — 4/256 = 2^-6;  XOR — 1


### 3-xulosa. SBZ variantlari 8-raunddan keyin differensial chegaraga erishadi

Misol: A ilova kaliti va xabari; ikkinchi blokning 0- va 9-elementlariga MSB ayirmasi kiritiladi.
To‘rttala variantda ayirma 0-nuqtadan 9-raund yutishigacha kuzatiladi va yutish nuqtalaridagi faol elementlar
asosida iz ehtimolligi chegarasi hisoblanadi (bayt boshiga: SBZ — 2⁻⁶, Z — 2⁻²·⁶⁸, XOR — 1).

In [10]:
D3 = {}
for p, bz in [(256, 'std'), (256, 'sbz'), (16, 'std'), (16, 'sbz')]:
    c = Cfg(p, bz); msb = p // 2
    if p == 256: h, b, kk = BLOK1, BLOK2, KALIT
    else:
        kir16 = kirish_bloklari(XABAR, 16); h, b, kk = kir16[0], kir16[1], K16
    b2 = b.copy(); b2[0] ^= msb; b2[9] ^= msb
    t1, t2 = iz(c, h, b, kk), iz(c, h, b2, kk)
    rows = [(nm, j, int(((x1 ^ x2) != 0).sum()), x1 ^ x2) for (nm, j, x1, k1), (_, _, x2, k2) in zip(t1, t2)]
    get = lambda nm: [r_ for r_ in rows if r_[0] == nm]
    q0, bz0 = get('Qosh0')[0], get('BZ0')[0]
    a8 = get('Qosh_inj8')[0][2]; a9 = get('Qosh_inj9')[0][2]
    per = {(256, 'std'): 2.68, (256, 'sbz'): 6.0, (16, 'std'): 0.0, (16, 'sbz'): 6.0}[(p, bz)]
    unit = 1 if p == 256 else 2        # 128 bitli rejimda bitta SBZ bayti = 2 ta yarim bayt
    r1 = [r_[2] for r_ in rows if r_[0] == 'Tuz' and r_[1] == 1][0]
    D3[f'{p}_{bz}'] = dict(faol_q0=q0[2], dbz0=hx(bz0[3]), faol_bz0=bz0[2], r1=r1, a8=a8, a9=a9,
                          chegara8=round(-per * a8 / unit, 1), chegara89=round(-per * (a8 + a9) / unit, 1))
    print(f"\n{p}-{bz}: Qo‘sh(0) dan keyin holatdagi faol: {q0[2]}; yutishdan keyin: {bz0[2]}; 1-raund: {r1}; 8-nuqtaga kirishda: {a8}; 9-nuqtaga kirishda: {a9}")
    chiqar("  yutishdan (0-nuqta) keyingi Δholat:", bz0[3])
    print(f"  chegara: 8-nuqtadan keyin log2 p <= {D3[f'{p}_{bz}']['chegara8']}, 8- va 9-nuqtalardan keyin <= {D3[f'{p}_{bz}']['chegara89']}")
R['D3'] = D3


256-std: Qo‘sh(0) dan keyin holatdagi faol: 0; yutishdan keyin: 2; 1-raund: 31; 8-nuqtaga kirishda: 32; 9-nuqtaga kirishda: 32
  yutishdan (0-nuqta) keyingi Δholat:
    3 0 0 0 0 0 0 0
    0 1 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
  chegara: 8-nuqtadan keyin log2 p <= -85.8, 8- va 9-nuqtalardan keyin <= -171.5

256-sbz: Qo‘sh(0) dan keyin holatdagi faol: 0; yutishdan keyin: 2; 1-raund: 32; 8-nuqtaga kirishda: 31; 9-nuqtaga kirishda: 32
  yutishdan (0-nuqta) keyingi Δholat:
    ea 00 00 00 00 00 00 00
    00 74 00 00 00 00 00 00
    00 00 00 00 00 00 00 00
    00 00 00 00 00 00 00 00
  chegara: 8-nuqtadan keyin log2 p <= -186.0, 8- va 9-nuqtalardan keyin <= -378.0

16-std: Qo‘sh(0) dan keyin holatdagi faol: 0; yutishdan keyin: 2; 1-raund: 29; 8-nuqtaga kirishda: 31; 9-nuqtaga kirishda: 26
  yutishdan (0-nuqta) keyingi Δholat:
    8 0 0 0 0 0 0 0
    0 8 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
  chegara: 8-nuqtadan keyin log2 p <= -0.0, 8- va 9-nuqtalardan keyi

### 4-xulosa. Aralash yadrosi orqali ayirma ehtimollik 1 bilan so‘nadi

M(k0) matritsasida nol ustun bo‘lgan kalit topiladi. Shu ustunga mos holat baytiga MSB ayirmasi kiritilsa,
Aralash dan keyin ayirma to‘liq yo‘qoladi va keyingi barcha holatlar bir xil bo‘ladi (ichki kolliziya).

In [11]:
rng = np.random.default_rng(4)
for t in range(500):
    kk = kalitlar(rng, 1, 256)[0]; Mm = M_aralash(kk); zc = [int(i) for i in np.nonzero(Mm.sum(0) == 0)[0]]
    if zc: break
pos_ = zc[0]; s_, u_ = pos_ // 8, pos_ % 8
print("Kalit:", bytes(int(v) for v in kk).hex()); print(f"M(k0) ning nol ustunlari: {[pos(i) for i in zc]};  tanlangan pozitsiya holat[{s_},{u_}]")
kcol = [int(kk[8*i + u_]) for i in range(4)]
print(f"Shu ustundagi ke[i,{u_}] qiymatlari: {[hex(v) for v in kcol]}  ->  juftliklari {[v % 2 for v in kcol]}")
h = rng.integers(0, 256, 32); h2 = h.copy(); h2[pos_] ^= 0x80
c = Cfg(256, 'sbz'); jr = juft_iz(c, h, h2, BLOK2, kk)
d_ar = [t for t in jr if t[0] == 'Aralash' and t[1] == 1][0][2]
chiqar("1-raund Aralash dan keyingi Δholat:", d_ar)
oxir = [t for t in jr if t[1] == 10][-1]
print("10-raunddan keyin Δholat nolmi:", bool((oxir[2] == 0).all()), "; Δke nolmi:", bool((oxir[3] == 0).all()))
R['D4'] = dict(kalit=bytes(int(v) for v in kk).hex(), nol_ustunlar=[pos(i) for i in zc], pozitsiya=[s_, u_], kcol=kcol,
               d_aralash=hx(d_ar), oxir_nol=bool((oxir[2] == 0).all() and (oxir[3] == 0).all()))

Kalit: b8f0e182f0f9f814749b4860a0cd942caddf388b56e60f7ae56e23c9f6fbee5e
M(k0) ning nol ustunlari: ['[1,4]', '[1,7]'];  tanlangan pozitsiya holat[1,4]
Shu ustundagi ke[i,4] qiymatlari: ['0xf0', '0xa0', '0x56', '0xf6']  ->  juftliklari [0, 0, 0, 0]
1-raund Aralash dan keyingi Δholat:
    0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0
10-raunddan keyin Δholat nolmi: True ; Δke nolmi: True


## 3. Algebraik kriptotahlil

### 1-xulosa. Yagona yuqori darajali element — yutish nuqtasidagi o‘zgartirish

In [12]:
yb = (Z[hh] << 4) | Z[nn]
deg = dict(XOR=[1]*4, Z=[anf_deg((Z >> j) & 1, 8) for j in range(4)], BaytZichlash_256=[anf_deg((yb >> j) & 1, 16) for j in range(8)],
           S=[anf_deg((S >> j) & 1, 8) for j in range(8)], SBZ=[anf_deg((y >> j) & 1, 16) for j in range(8)],
           Qosh_element=[anf_deg((f >> j) & 1, 8) for j in range(8)])
for kk_, v in deg.items(): print(f"{kk_:17s} chiqish bitlari darajalari: {v}")
print(f"(Qo‘sh — holat[0,0] elementi, parametrlar: {R['L1']['param']})")
row = Mk[0]; terms = [f"h[{i//8},{i%8}]" for i in np.nonzero(row)[0]]
print("\nA ilova kaliti uchun Aralash 0-qatlam tenglamasi:  h'[0,0]₀ =", " ⊕ ".join(terms) if terms else "0")
R['A1'] = dict(deg=deg, aralash_tengl=terms)

XOR               chiqish bitlari darajalari: [1, 1, 1, 1]
Z                 chiqish bitlari darajalari: [6, 6, 6, 6]
BaytZichlash_256  chiqish bitlari darajalari: [6, 6, 6, 6, 6, 6, 6, 6]
S                 chiqish bitlari darajalari: [7, 7, 7, 7, 7, 7, 7, 7]
SBZ               chiqish bitlari darajalari: [13, 13, 13, 13, 13, 13, 13, 13]
Qosh_element      chiqish bitlari darajalari: [1, 1, 8, 3, 8, 5, 8, 8]
(Qo‘sh — holat[0,0] elementi, parametrlar: {'R': 200, 'A1': 73, 'B1': 201, 'A2': 47, 'B2': 245})

A ilova kaliti uchun Aralash 0-qatlam tenglamasi:  h'[0,0]₀ = h[1,1] ⊕ h[2,2]


### 2-xulosa. Tizim hajmini Qo‘sh belgilaydi, SBZ esa darajani oshiradi

A ilova ma’lumotlari bo‘yicha 0-nuqtadagi Qo‘sh o‘zgartirishining holat[0,0] elementi uchun halqa
tenglamalarini haqiqiy sonlar bilan tekshiramiz, so‘ng bit darajasidagi tenglamalar sonini taqsimlaymiz.

In [13]:
e = 0; x0 = int(BLOK1[e]); x1 = int(q['x1'][e]); x2 = int(q['x2'][e]); x3 = int(q['x3'][e])
print(f"J1h = Σ holat = {q['J1h']},  J1hn = Σ holatn = {q['J1hn']},  K1h = {q['K1h']},  K2h = {q['K2h']},  J2hn = {q['J2hn']}")
print(f"A1 = J1hn + 2·hn[0,0] + Δ = {A1},   B1 = K1h − 2·h[0,0] = {B1},   A2 = {A2},   B2 = {B2},   R = {Rv}")
t1 = (x1 * (B1 + Rv * x0) + A1 * x0) % 256
t2 = (x2 * (B2 + Rv * x1) + A2 * x1) % 256
t3 = (x3 * (B2 + Rv * x2) + A2 * x2) % 256
print(f"x = {x0:#04x} -> x⁻¹ = {x1:#04x} -> x⁻² = {x2:#04x} -> x⁻³ = {x3:#04x}")
print(f"1-bosqich: x⁻¹·(B1 + R·x) + A1·x ≡ {t1} (mod 256)")
print(f"2-bosqich: x⁻²·(B2 + R·x⁻¹) + A2·x⁻¹ ≡ {t2} (mod 256)")
print(f"3-bosqich: x⁻³·(B2 + R·x⁻²) + A2·x⁻² ≡ {t3} (mod 256)")
import algebraik
E, LC = algebraik.tenglamalar_soni(); e256 = E['256_sbz']; tot = e256['raund'][0]
print(f"\nBitta raund (256, yutishsiz): {tot} o‘zgaruvchi;  2×Qo‘sh = {2*e256['qosh'][0]} ({100*2*e256['qosh'][0]/tot:.1f}%),"
      f" Aralash = {e256['aralash'][0]}, TuzilmaKalit = {e256['tuzilma'][0]};  SBZ qatlami (o‘zg., tengl., kvadrat) = {e256['yutish']}")
R['A2'] = dict(x=[x0, x1, x2, x3], A1=A1, B1=B1, A2=A2, B2=B2, J1h=q['J1h'], J1hn=q['J1hn'], K1h=q['K1h'], tekshiruv=[t1, t2, t3],
               raund=tot, qosh2=2*e256['qosh'][0], aralash=e256['aralash'][0], tuz=e256['tuzilma'][0], sbz=list(e256['yutish']),
               bz_std=list(E['256_std']['yutish']))

J1h = Σ holat = 153,  J1hn = Σ holatn = 109,  K1h = 149,  K2h = 133,  J2hn = 237
A1 = J1hn + 2·hn[0,0] + Δ = 73,   B1 = K1h − 2·h[0,0] = 201,   A2 = 47,   B2 = 245,   R = 200
x = 0x66 -> x⁻¹ = 0xba -> x⁻² = 0x12 -> x⁻³ = 0x8a
1-bosqich: x⁻¹·(B1 + R·x) + A1·x ≡ 0 (mod 256)
2-bosqich: x⁻²·(B2 + R·x⁻¹) + A2·x⁻¹ ≡ 0 (mod 256)
3-bosqich: x⁻³·(B2 + R·x⁻²) + A2·x⁻² ≡ 0 (mod 256)

Bitta raund (256, yutishsiz): 243108 o‘zgaruvchi;  2×Qo‘sh = 222164 (91.4%), Aralash = 20208, TuzilmaKalit = 736;  SBZ qatlami (o‘zg., tengl., kvadrat) = (512, 2496, 2496)


### 3-xulosa. Standart-128 da 0-qatlam chiziqli: 80 ta so‘rov bilan 32 bit bashorat qilinadi

In [14]:
rng = np.random.default_rng(31); k16b = np.array(list(KAT_KALIT[:16]))
tr_m = rng.integers(0, 256, (80, 16)); te_m = rng.integers(0, 256, (300, 16))
feats = lambda m: np.concatenate([to_units(m, 16) % 2, np.ones((len(m), 1), dtype=np.int64)], 1)
A3 = {}
for bz in ('std', 'sbz'):
    c = Cfg(16, bz); Hf = lambda m: hash_msgs(m, np.tile(k16b, (len(m), 1)), c)
    Wm = algebraik.gf2_solve(feats(tr_m), Hf(tr_m) % 2); pred = (feats(te_m) @ Wm) % 2; act = Hf(te_m) % 2
    A3[bz] = dict(toliq=float((pred == act).all(1).mean()), bit=round(float((pred == act).mean()), 4),
                  rangA=rank2(Wm[:32].T.astype(np.int64)), c_vektor=''.join(map(str, Wm[32])),
                  misollar=[(bytes(te_m[i].tolist()).hex(), ''.join(map(str, act[i])), ''.join(map(str, pred[i]))) for i in range(3)])
    print(f"\n{bz}: 32 bitning hammasi to‘g‘ri — {A3[bz]['toliq']:.3f};  bitlar bo‘yicha — {A3[bz]['bit']:.3f};  rang(A) = {A3[bz]['rangA']}")
    for m_, a_, p_ in A3[bz]['misollar']: print(f"   xabar {m_}\n     haqiqiy:  {a_}\n     bashorat: {p_}")
R['A3'] = A3


std: 32 bitning hammasi to‘g‘ri — 1.000;  bitlar bo‘yicha — 1.000;  rang(A) = 25
   xabar 3113a609017aa8c3ad16fa9294101a75
     haqiqiy:  01100011101000110000010111000000
     bashorat: 01100011101000110000010111000000
   xabar 3e7f7877070e208ef96876e2b0437dce
     haqiqiy:  11000011111001111010010001111001
     bashorat: 11000011111001111010010001111001
   xabar 612a4d165c8d8f61add41c17d72db1fc
     haqiqiy:  11001010010110010010010000000010
     bashorat: 11001010010110010010010000000010

sbz: 32 bitning hammasi to‘g‘ri — 0.000;  bitlar bo‘yicha — 0.569;  rang(A) = 25
   xabar 3113a609017aa8c3ad16fa9294101a75
     haqiqiy:  01000010100101010101010111100111
     bashorat: 10101000011000101000000110101101
   xabar 3e7f7877070e208ef96876e2b0437dce
     haqiqiy:  00000001100000010101010111101101
     bashorat: 11100011011011111100010000000000
   xabar 612a4d165c8d8f61add41c17d72db1fc
     haqiqiy:  01001000101000011100010000110000
     bashorat: 01100010001010100111000000011001


### 4-xulosa. SBZ variantlari algebraik chegaraga 0-nuqtadagi yutishdan keyinoq erishadi

Kub testi misoli: xabarning 0-baytidan 7 ta bit tanlanadi va 2⁷ ta qiymat bo‘yicha holat bitlari
yig‘indisi hisoblanadi (r — 0-nuqtadan keyingi raundlar soni). Shuningdek, talab qilinadigan daraja
linearizatsiya murakkabligi bo‘yicha aniqlanadi.

In [15]:
def kub(p, bz, r, poz, kalit, h, bl):
    c = Cfg(p, bz); ub = 8 if p == 256 else 4; d = len(poz); N = 1 << d; Bm = np.tile(bl, (N, 1))
    for t, q_ in enumerate(poz):
        u, b_ = q_ // ub, q_ % ub; Bm[:, u] &= ~(1 << b_); Bm[:, u] |= ((np.arange(N) >> t) & 1) << b_
    K = np.tile(kalit, (N, 1)); out, _ = iteration(np.tile(h, (N, 1)), Bm, K.copy(), K, c, rounds=r)
    s_ = np.bitwise_xor.reduce(out, axis=0); return np.array([(s_ >> j) & 1 for j in range(ub)]).T
poz = [0, 1, 2, 3, 4, 5, 6]; A4 = {}   # xabarning 0-baytidagi 7 ta bit (d = 7)
for p, bz in [(256, 'std'), (256, 'sbz'), (16, 'std'), (16, 'sbz')]:
    if p == 256: kk_, h_, b_ = KALIT, BLOK1, BLOK2
    else: kir16 = kirish_bloklari(XABAR, 16); kk_, h_, b_ = K16, kir16[0], kir16[1]
    for r in (0, 1, 3):
        sm = kub(p, bz, r, poz, kk_, h_, b_)
        A4[f'{p}_{bz}_r{r}'] = dict(nol=int((sm == 0).sum()), jami=int(sm.size), qatlam0_nol=int((sm[:, 0] == 0).sum()))
        print(f"{p}-{bz}, r={r}: yig‘indisi nol bo‘lgan bitlar {int((sm==0).sum())}/{sm.size};  0-qatlamda {int((sm[:,0]==0).sum())}/32")
lc = lambda nv, d: 2.8 * math.log2(sum(math.comb(nv, i) for i in range(d + 1)))
dmin128 = min(d for d in range(1, 30) if lc(128, d) >= 64); dmin256 = min(d for d in range(1, 30) if lc(256, d) >= 128)
print(f"\nTalab qilinadigan daraja: 128 bitli rejimda d >= {dmin128} (2^{lc(128,dmin128):.1f}), 256 bitli rejimda d >= {dmin256} (2^{lc(256,dmin256):.1f})")
print(f"Yutishdagi darajalar: XOR = 1 (2^{lc(128,1):.1f}),  Z = 6 (2^{lc(256,6):.1f}),  SBZ = 13 (2^{lc(256,13):.1f} va 2^{lc(128,13):.1f})")
R['A4'] = dict(kub=A4, poz=poz, dmin128=dmin128, dmin256=dmin256, lc={f'{nv}_{d}': round(lc(nv, d), 1) for nv in (128, 256) for d in (1, 4, 6, 7, 8, 13)})

256-std, r=0: yig‘indisi nol bo‘lgan bitlar 124/256;  0-qatlamda 17/32
256-std, r=1: yig‘indisi nol bo‘lgan bitlar 120/256;  0-qatlamda 14/32
256-std, r=3: yig‘indisi nol bo‘lgan bitlar 114/256;  0-qatlamda 12/32
256-sbz, r=0: yig‘indisi nol bo‘lgan bitlar 126/256;  0-qatlamda 15/32
256-sbz, r=1: yig‘indisi nol bo‘lgan bitlar 137/256;  0-qatlamda 14/32
256-sbz, r=3: yig‘indisi nol bo‘lgan bitlar 129/256;  0-qatlamda 16/32
16-std, r=0: yig‘indisi nol bo‘lgan bitlar 127/128;  0-qatlamda 32/32
16-std, r=1: yig‘indisi nol bo‘lgan bitlar 103/128;  0-qatlamda 32/32
16-std, r=3: yig‘indisi nol bo‘lgan bitlar 80/128;  0-qatlamda 32/32
16-sbz, r=0: yig‘indisi nol bo‘lgan bitlar 124/128;  0-qatlamda 31/32
16-sbz, r=1: yig‘indisi nol bo‘lgan bitlar 74/128;  0-qatlamda 30/32
16-sbz, r=3: yig‘indisi nol bo‘lgan bitlar 72/128;  0-qatlamda 26/32

Talab qilinadigan daraja: 128 bitli rejimda d >= 4 (2^65.5), 256 bitli rejimda d >= 8 (2^136.0)
Yutishdagi darajalar: XOR = 1 (2^19.6),  Z = 6 (2^107.7),  S

In [16]:
json.dump(R, open('misollar_natijalari.json', 'w'), ensure_ascii=False, indent=1,
          default=lambda o: o.tolist() if hasattr(o, 'tolist') else (int(o) if isinstance(o, (np.integer,)) else str(o)))
print("Natijalar misollar_natijalari.json fayliga saqlandi.")

Natijalar misollar_natijalari.json fayliga saqlandi.
